In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs (files are in current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

# Convert timestamp to timezone-aware datetimes in UTC then to Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Global time ranges for each file (compact)
metric_time_range = {
    'global_min_ts': df_metric['ts'].min(),
    'global_max_ts': df_metric['ts'].max()
}
trace_time_range = {
    'global_min_ts': df_trace['ts'].min(),
    'global_max_ts': df_trace['ts'].max()
}

# Aggregation helpers: compute count, min/max timestamps, p95, p5, mean, std
def aggregate_metric(df, group_cols, value_col):
    # group and compute statistics
    grp = df.groupby(group_cols)
    agg = grp.agg(
        count=(value_col, 'count'),
        ts_min=('ts', 'min'),
        ts_max=('ts', 'max'),
        p95=(value_col, lambda x: x.quantile(0.95)),
        p5=(value_col, lambda x: x.quantile(0.05)),
        mean=(value_col, 'mean'),
        std=(value_col, 'std'),
    ).reset_index()
    # Round numeric percentiles to at least 4 decimal places
    for c in ['p95', 'p5', 'mean', 'std']:
        agg[c] = agg[c].round(4)
    # Sort by count desc and limit to top 50
    agg = agg.sort_values(by='count', ascending=False).head(50).reset_index(drop=True)
    return agg

metric_aggregates = aggregate_metric(df_metric, ['cmdb_id', 'kpi_name'], 'value')
trace_aggregates = aggregate_metric(df_trace, ['cmdb_id', 'trace_name'], 'value')

# Keep outputs compact: show aggregated tables (<=50 rows) and the two file time ranges
metric_aggregates, trace_aggregates, metric_time_range, trace_time_range

```
Out[1]:
```
```python
summary = (
    "Summary of aggregations:\n"
    "- metric.csv global time range (Asia/Shanghai UTC+8): 2020-05-24 01:00:00+08:00 -> 2020-05-24 02:00:00+08:00.\n"
    "- trace.csv global time range (Asia/Shanghai UTC+8): 2020-05-24 01:30:00+08:00 -> 2020-05-24 01:59:00+08:00.\n\n"
    "Metric aggregates (top 50 groups by count):\n"
    "- Returned 50 groups, max count = 61. Examples of top groups:\n"
    "  * db_001 / CPU_free_pct: count=61, p95=98.9751, p5=97.7938, mean=98.5584, std=0.4186\n"
    "  * os_017 / Send_total: count=61, p95=0.3332, p5=0.2650, mean=0.3040, std=0.0463\n"
    "  * os_022 / Outgoing_network_traffic: count=61, p95=0.0850, p5=0.0524, mean=0.0770, std=0.0135\n"
    "  * db_007 / MEM_real_util: count=61, p95=82.2500, p5=81.9700, mean=82.2087, std=0.0739\n"
    "  * db_008 / CPU_free_pct: count=61, p95=97.7767, p5=96.4428, mean=97.1646, std=0.4432\n"
    "- Many os_001 KPIs appear with count=60 in the top-50 list.\n\n"
    "Trace aggregates (top 50 groups by count):\n"
    "- Returned 50 groups, max count = 30. Examples of top groups:\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.duration_mean: count=30, p95=0.7977, p5=0.4916, mean=0.6579, std=0.1048\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.duration_p95: count=30, p95=2.4538, p5=1.0478, mean=1.8656, std=0.4531\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.error_rate: count=30, p95=0.0000, p5=0.0000, mean=0.0000, std=0.0000\n"
    "  * docker_001 / trace.from_os_022.remoteprocess.row_count: count=30, p95=155.1000, p5=125.7000, mean=142.2000, std=13.3272\n"
    "  * docker_001 / trace.self.csf.duration_mean: count=30, p95=0.0785, p5=0.0642, mean=0.0712, std=0.0048\n"
    "- docker_002 groups also appear frequently at count=30 in the top-50 list.\n\n"
    "Notes:\n"
    "- Both tables were limited to 50 rows as requested. Numeric percentiles shown with >=4 decimal precision.\n"
    "- For full per-group details refer to the metric_aggregates and trace_aggregates tables produced earlier."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                  kpi_name  count                    ts_min                    ts_max      p95       p5     mean     std
0   db_001              CPU_free_pct     61 2020-05-24 01:00:00+08:00 2020-05-24 02:00:00+08:00  98.9751  97.7938  98.5584  0.4186
1   os_017                Send_total     61 2020-05-24 01:00:00+08:00 2020-05-24 02:00:00+08:00   0.3332   0.2650   0.3040  0.0463
2   os_022  Outgoing_network_traffic     61 2020-05-24 01:00:00+08:00 2020-05-24 02:00:00+08:00   0.0850   0.0524   0.0770  0.0135
3   db_007             MEM_real_util     61 2020-05-24 01:00:00+08:00 2020-05-24 02:00:00+08:00  82.2500  81.9700  82.2087  0.0739
4   db_008              CPU_free_pct     61 2020-05-24 01:00:00+08:00 2020-05-24 02:00:00+08:00  97.7767  96.4428  97.1646  0.4432
..     ...                       ...    ...                       ...                       ...      ...      ...      ...     ...
45  os_001     Processor_load_15_min     60 2020-05-24 01:00:00+08:00 2020-05-24 01:59:00+08:00   0.0684   0.0458   0.0554  0.0070
46  os_001                 ICMP_ping     60 2020-05-24 01:00:00+08:00 2020-05-24 01:59:00+08:00   1.0000   1.0000   1.0000  0.0000
47  os_001  Incoming_network_traffic     60 2020-05-24 01:00:00+08:00 2020-05-24 01:59:00+08:00   0.0182   0.0147   0.0165  0.0017
48  os_001               Disk_rd_ios     60 2020-05-24 01:00:00+08:00 2020-05-24 01:59:00+08:00   0.0000   0.0000   0.0000  0.0000
49  os_001               Disk_rd_kbs     60 2020-05-24 01:00:00+08:00 2020-05-24 01:59:00+08:00   0.0000   0.0000   0.0000  0.0000

[50 rows x 9 columns],        cmdb_id                                       trace_name  count                    ts_min                    ts_max        p95         p5       mean       std
0   docker_001    trace.from_os_022.remoteprocess.duration_mean     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     0.7977     0.4916     0.6579    0.1048
1   docker_001     trace.from_os_022.remoteprocess.duration_p95     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     2.4538     1.0478     1.8656    0.4531
2   docker_001       trace.from_os_022.remoteprocess.error_rate     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     0.0000     0.0000     0.0000    0.0000
3   docker_001        trace.from_os_022.remoteprocess.row_count     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00   155.1000   125.7000   142.2000   13.3272
4   docker_001                     trace.self.csf.duration_mean     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     0.0785     0.0642     0.0712    0.0048
..         ...                                              ...    ...                       ...                       ...        ...        ...        ...       ...
45  docker_002                    trace.self.local.duration_p95     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     0.0960     0.0894     0.0930    0.0022
46  docker_002                      trace.self.local.error_rate     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     0.0714     0.0000     0.0120    0.0268
47  docker_002                       trace.self.local.row_count     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00  2291.1000  1540.0000  2029.5333  203.8259
48  docker_002  trace.to_docker_007.remoteprocess.duration_mean     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     0.0471     0.0441     0.0454    0.0009
49  docker_002   trace.to_docker_007.remoteprocess.duration_p95     30 2020-05-24 01:30:00+08:00 2020-05-24 01:59:00+08:00     0.0560     0.0530     0.0541    0.0011

[50 rows x 9 columns], {'global_min_ts': Timestamp('2020-05-24 01:00:00+0800', tz='Asia/Shanghai'), 'global_max_ts': Timestamp('2020-05-24 02:00:00+0800', tz='Asia/Shanghai')}, {'global_min_ts': Timestamp('2020-05-24 01:30:00+0800', tz='Asia/Shanghai'), 'global_max_ts': Timestamp('2020-05-24 01:59:00+0800', tz='Asia/Shanghai')})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing variables: df_metric, metric_aggregates, tz
import pandas as pd
import numpy as np

# Define incident window in Asia/Shanghai (UTC+8)
window_start = pd.Timestamp('2020-05-24 01:30:00', tz=tz)
window_end = pd.Timestamp('2020-05-24 02:00:00', tz=tz)

# Filter metric records to the incident window (inclusive)
df_metric_window = df_metric[(df_metric['ts'] >= window_start) & (df_metric['ts'] <= window_end)].copy()

# Prepare global thresholds from metric_aggregates (p95 and p5 computed globally earlier)
# Ensure needed columns exist and have numeric types
thr = metric_aggregates[['cmdb_id', 'kpi_name', 'p95', 'p5', 'mean', 'std']].copy()
for c in ['p95', 'p5', 'mean', 'std']:
    thr[c] = pd.to_numeric(thr[c], errors='coerce')

# Merge thresholds into windowed data; drop rows lacking global thresholds (cannot evaluate anomalies)
df_w = df_metric_window.merge(thr, on=['cmdb_id', 'kpi_name'], how='left')
df_w = df_w.dropna(subset=['p95', 'p5'])  # keep only groups with global thresholds

# Compute anomaly flag per row using global thresholds: value >= p95 OR value <= p5
df_w['is_anomaly'] = (df_w['value'] >= df_w['p95']) | (df_w['value'] <= df_w['p5'])

# Create a column that is ts when anomaly else NaT for earliest anomaly calculation
df_w['anomaly_ts'] = df_w['ts'].where(df_w['is_anomaly'], pd.NaT)

# Aggregate per (cmdb_id, kpi_name)
agg = df_w.groupby(['cmdb_id', 'kpi_name']).agg(
    anomaly_count=('is_anomaly', 'sum'),
    total_in_window=('value', 'count'),
    earliest_anomaly_ts=('anomaly_ts', 'min'),
    p95=('p95', 'first'),
    p5=('p5', 'first'),
    mean=('mean', 'first'),
    std=('std', 'first'),
).reset_index()

# Keep only groups with at least one anomaly
agg = agg[agg['anomaly_count'] > 0].copy()

# Round numeric columns to at least 4 decimals
for c in ['p95', 'p5', 'mean', 'std']:
    agg[c] = agg[c].round(4)

# Sort by anomaly_count descending and limit to top 20
anomalies_summary = agg.sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Keep output compact with required columns and types
anomalies_summary = anomalies_summary[[
    'cmdb_id', 'kpi_name', 'anomaly_count', 'total_in_window', 'earliest_anomaly_ts', 'p95', 'p5', 'mean', 'std'
]]

anomalies_summary

```
Out[2]:
```
summary_lines = []
window_start = "2020-05-24 01:30:00+08:00"
window_end = "2020-05-24 02:00:00+08:00"

summary_lines.append(f"Incident window (Asia/Shanghai UTC+8): {window_start} -> {window_end} (inclusive).")
n_groups = anomalies_summary.shape[0]
summary_lines.append(f"Found {n_groups} groups with >=1 anomaly in-window (groups with zero anomalies excluded).")
summary_lines.append("")
summary_lines.append("Top groups by anomaly_count (showing up to 5):")

top_n = min(5, n_groups)
for i in range(top_n):
    r = anomalies_summary.iloc[i]
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    ac = int(r['anomaly_count'])
    tot = int(r['total_in_window'])
    ets = r['earliest_anomaly_ts']
    p95 = f"{r['p95']:.4f}"
    p5 = f"{r['p5']:.4f}"
    mean = f"{r['mean']:.4f}"
    std = f"{r['std']:.4f}"
    summary_lines.append(
        f"{i+1}) {cmdb} / {kpi}: anomaly_count={ac}, total_in_window={tot}, earliest_anomaly_ts={ets}, "
        f"p95={p95}, p5={p5}, mean={mean}, std={std}"
    )

summary_lines.append("")
summary_lines.append("Notable observations from the top 20 (not exhaustive):")
summary_lines.append("- Several disk read metrics (e.g., Disk_rd_ios, Disk_rd_kbs) have anomalies spanning the entire window (anomaly_count equals total_in_window).")
summary_lines.append("- ICMP/agent ping metrics for multiple OS nodes also show anomalies across the full window (values at p95/p5 boundaries).")
summary_lines.append("- Some resource metrics (e.g., db_007 MEM_real_util) show fewer anomalies but with non-trivial values (see provided p95/p5/mean/std).")
summary_lines.append("")
summary_lines.append("If you want the full top-20 table formatted or exported, ask to display 'anomalies_summary'.")

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                  kpi_name  anomaly_count  total_in_window       earliest_anomaly_ts       p95        p5      mean      std
0    os_007               Disk_rd_ios             31               31 2020-05-24 01:30:00+08:00    0.0000    0.0000    0.0000   0.0000
1    os_001               Disk_rd_kbs             30               30 2020-05-24 01:30:00+08:00    0.0000    0.0000    0.0000   0.0000
2    os_001                 ICMP_ping             30               30 2020-05-24 01:30:00+08:00    1.0000    1.0000    1.0000   0.0000
3    os_002                 ICMP_ping             30               30 2020-05-24 01:30:00+08:00    1.0000    1.0000    1.0000   0.0000
4    os_002              Disk_io_util             30               30 2020-05-24 01:30:00+08:00    0.0000    0.0000    0.2324   1.3798
5    os_002                Agent_ping             30               30 2020-05-24 01:30:00+08:00    1.0000    1.0000    1.0000   0.0000
6    os_001               Disk_rd_ios             30               30 2020-05-24 01:30:00+08:00    0.0000    0.0000    0.0000   0.0000
7    os_018                Agent_ping             30               30 2020-05-24 01:30:00+08:00    1.0000    1.0000    1.0000   0.0000
8    os_017               Disk_rd_ios             30               30 2020-05-24 01:30:00+08:00    0.0000    0.0000    0.0000   0.0000
9    os_018               Disk_rd_ios             30               30 2020-05-24 01:30:00+08:00    0.0000    0.0000    0.0000   0.0000
10   os_018              Disk_io_util             30               30 2020-05-24 01:30:00+08:00    0.0000    0.0000    0.0000   0.0000
11   os_017                 ICMP_ping             30               30 2020-05-24 01:30:00+08:00    1.0000    1.0000    1.0000   0.0000
12   os_020              Disk_io_util             29               31 2020-05-24 01:30:00+08:00    0.0055    0.0000    0.0005   0.0015
13   os_018                Disk_await             29               30 2020-05-24 01:30:00+08:00   20.6425    0.0000    3.0413   6.9147
14  osb_001           app.succee_rate             28               31 2020-05-24 01:30:00+08:00    1.0000    0.0000    0.9170   0.2745
15   os_001                   Page_po             14               30 2020-05-24 01:30:00+08:00    8.0000    5.0000    6.5333   1.0651
16   os_015  Incoming_network_traffic             14               31 2020-05-24 01:30:00+08:00    0.0024    0.0023    0.0022   0.0006
17   db_007             MEM_real_util             13               31 2020-05-24 01:31:00+08:00   82.2500   81.9700   82.2087   0.0739
18   os_012              Disk_io_util             12               31 2020-05-24 01:33:00+08:00    2.0000    0.5000    0.9732   0.5129
19   os_001                  ss_total              7               30 2020-05-24 01:31:00+08:00  167.0500  122.0000  140.1833  15.7518```
```

In [ ]:
# In[3]:

# Reuse existing variables: df_trace, trace_aggregates, tz, window_start, window_end
import pandas as pd

# Ensure window_start/window_end exist (they were defined previously). If not, define them.
try:
    window_start, window_end
except NameError:
    window_start = pd.Timestamp('2020-05-24 01:30:00', tz=tz)
    window_end = pd.Timestamp('2020-05-24 02:00:00', tz=tz)

# Filter trace records to the incident window (inclusive)
df_trace_window = df_trace[(df_trace['ts'] >= window_start) & (df_trace['ts'] <= window_end)].copy()

# Prepare global thresholds from trace_aggregates (p95 and p5 computed globally earlier)
thr_trace = trace_aggregates[['cmdb_id', 'trace_name', 'p95', 'p5', 'mean', 'std']].copy()
for c in ['p95', 'p5', 'mean', 'std']:
    thr_trace[c] = pd.to_numeric(thr_trace[c], errors='coerce')

# Merge thresholds into windowed trace data; drop rows lacking global thresholds
df_tw = df_trace_window.merge(thr_trace, on=['cmdb_id', 'trace_name'], how='left')
df_tw = df_tw.dropna(subset=['p95', 'p5'])

# Compute anomaly flag per row using global thresholds: value >= p95 OR value <= p5
df_tw['is_anomaly'] = (df_tw['value'] >= df_tw['p95']) | (df_tw['value'] <= df_tw['p5'])
df_tw['anomaly_ts'] = df_tw['ts'].where(df_tw['is_anomaly'], pd.NaT)

# Aggregate per (cmdb_id, trace_name)
agg_trace = df_tw.groupby(['cmdb_id', 'trace_name']).agg(
    anomaly_count=('is_anomaly', 'sum'),
    total_in_window=('value', 'count'),
    earliest_anomaly_ts=('anomaly_ts', 'min'),
    p95=('p95', 'first'),
    p5=('p5', 'first'),
    mean=('mean', 'first'),
    std=('std', 'first'),
).reset_index()

# Keep only groups with at least one anomaly
agg_trace = agg_trace[agg_trace['anomaly_count'] > 0].copy()

# Round numeric columns to at least 4 decimals
for c in ['p95', 'p5', 'mean', 'std']:
    agg_trace[c] = agg_trace[c].round(4)

# Sort by anomaly_count descending and limit to top 20
trace_anomalies_summary = agg_trace.sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Select and order required columns
trace_anomalies_summary = trace_anomalies_summary[[
    'cmdb_id', 'trace_name', 'anomaly_count', 'total_in_window', 'earliest_anomaly_ts', 'p95', 'p5', 'mean', 'std'
]]

trace_anomalies_summary

```
Out[3]:
```
```python
# Summarize trace anomaly results in plain English using existing `trace_anomalies_summary`
top_n = min(5, trace_anomalies_summary.shape[0])
lines = []
lines.append("Incident window (Asia/Shanghai UTC+8): 2020-05-24 01:30:00+08:00 -> 2020-05-24 02:00:00+08:00 (inclusive).")
lines.append(f"Found {trace_anomalies_summary.shape[0]} trace groups with >=1 anomaly in-window (groups with zero anomalies excluded).")
lines.append("")
lines.append("Top trace groups by anomaly_count (showing up to 5):")
for i in range(top_n):
    r = trace_anomalies_summary.iloc[i]
    cmdb = r['cmdb_id']
    name = r['trace_name']
    ac = int(r['anomaly_count'])
    tot = int(r['total_in_window'])
    ets = r['earliest_anomaly_ts']
    p95 = f"{r['p95']:.4f}"
    p5 = f"{r['p5']:.4f}"
    mean = f"{r['mean']:.4f}"
    std = f"{r['std']:.4f}"
    lines.append(f"{i+1}) {cmdb} / {name}: anomaly_count={ac}, total_in_window={tot}, earliest_anomaly_ts={ets}, p95={p95}, p5={p5}, mean={mean}, std={std}")

lines.append("")
lines.append("Notable observations:")
lines.append("- Many error_rate traces for docker_001 and docker_002 show anomalies spanning the entire window (anomaly_count = total_in_window = 30).")
lines.append("- Some error_rate series have global p95/p5 = 0.0000 which results in all points flagged (check whether zero thresholds are meaningful for those metrics).")
lines.append("- A few duration_p95 and row_count traces show partial anomalies (e.g., traces to/from other dockers and local.duration_p95 entries with smaller anomaly counts).")
lines.append("")
lines.append("If you want the full top-20 table, inspect the variable `trace_anomalies_summary`.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  anomaly_count  total_in_window       earliest_anomaly_ts        p95         p5       mean       std
0   docker_001                       trace.self.csf.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
1   docker_001      trace.from_os_022.remoteprocess.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
2   docker_001                      trace.self.jdbc.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0002    0.0008
3   docker_001                 trace.self.flyremote.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
4   docker_002                      trace.self.jdbc.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0001    0.0006
5   docker_002                 trace.self.flyremote.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
6   docker_002                       trace.self.csf.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
7   docker_002      trace.from_os_022.remoteprocess.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
8   docker_001    trace.to_docker_008.remoteprocess.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
9   docker_001    trace.to_docker_007.remoteprocess.error_rate             30               30 2020-05-24 01:30:00+08:00     0.0000     0.0000     0.0000    0.0000
10  docker_001                     trace.self.local.error_rate             28               30 2020-05-24 01:30:00+08:00     0.0714     0.0000     0.0120    0.0269
11  docker_002                     trace.self.local.error_rate             28               30 2020-05-24 01:30:00+08:00     0.0714     0.0000     0.0120    0.0268
12  docker_002  trace.to_docker_007.remoteprocess.duration_p95             10               30 2020-05-24 01:32:00+08:00     0.0560     0.0530     0.0541    0.0011
13  docker_001  trace.to_docker_008.remoteprocess.duration_p95              9               30 2020-05-24 01:30:00+08:00     0.0560     0.0538     0.0549    0.0008
14  docker_002                   trace.self.local.duration_p95              7               30 2020-05-24 01:31:00+08:00     0.0960     0.0894     0.0930    0.0022
15  docker_002                  trace.self.flyremote.row_count              5               30 2020-05-24 01:45:00+08:00   163.6500   110.0000   144.9667   14.5590
16  docker_001                   trace.self.local.duration_p95              5               30 2020-05-24 01:42:00+08:00     0.0970     0.0782     0.0908    0.0060
17  docker_002                        trace.self.csf.row_count              5               30 2020-05-24 01:45:00+08:00   654.6000   440.0000   579.8667   58.2360
18  docker_002                      trace.self.local.row_count              5               30 2020-05-24 01:45:00+08:00  2291.1000  1540.0000  2029.5333  203.8259
19  docker_002       trace.from_os_022.remoteprocess.row_count              5               30 2020-05-24 01:45:00+08:00   163.6500   110.0000   144.9667   14.5590```
```

In [ ]:
# In[4]:

# Aggregate anomalies_summary by cmdb_id and produce top contributors for top 3 cmdbs.
import pandas as pd

# Use existing anomalies_summary DataFrame from previous steps
# Aggregate per cmdb_id
agg_by_cmdb = (
    anomalies_summary
    .groupby('cmdb_id', dropna=False)
    .agg(
        total_anomaly_points=('anomaly_count', 'sum'),
        faulty_kpi_count=('kpi_name', 'nunique'),
        total_points_in_window=('total_in_window', 'sum'),
        earliest_anomaly_ts=('earliest_anomaly_ts', 'min'),
    )
    .reset_index()
    .sort_values(by='total_anomaly_points', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Prepare top KPI contributors for the top 3 cmdb_id
top3_cmdbs = agg_by_cmdb['cmdb_id'].head(3).tolist()

# For each top cmdb, take up to 5 KPIs with largest anomaly_count
top_kpis_for_top3 = (
    anomalies_summary[anomalies_summary['cmdb_id'].isin(top3_cmdbs)]
    .sort_values(['cmdb_id', 'anomaly_count'], ascending=[True, False])
    .groupby('cmdb_id')
    .head(5)
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count']]
    .reset_index(drop=True)
)

# Create a compact summary string listing top KPIs (<=5) per top-3 node
lines = []
for cmdb in top3_cmdbs:
    subset = top_kpis_for_top3[top_kpis_for_top3['cmdb_id'] == cmdb]
    if subset.empty:
        lines.append(f"{cmdb}: (no KPIs)")
        continue
    pairs = [f"{row.kpi_name}({int(row.anomaly_count)})" for row in subset.itertuples(index=False)]
    lines.append(f"{cmdb}: " + ", ".join(pairs))
top3_kpi_summary = " | ".join(lines)

# Display compact results: aggregated cmdb table (top 20) and KPI contributor table for top-3, plus the summary string
agg_by_cmdb, top_kpis_for_top3, top3_kpi_summary

```
Out[4]:
```
```python
summary = (
    "Aggregated metric anomalies by node (top results):\n"
    "- os_018: total_anomaly_points=119, faulty_kpi_count=4, total_points_in_window=120, earliest_anomaly_ts=2020-05-24 01:30:00+08:00\n"
    "- os_001: total_anomaly_points=111, faulty_kpi_count=5, total_points_in_window=150, earliest_anomaly_ts=2020-05-24 01:30:00+08:00\n"
    "- os_002: total_anomaly_points=90,  faulty_kpi_count=3, total_points_in_window=90,  earliest_anomaly_ts=2020-05-24 01:30:00+08:00\n"
    "- os_017: total_anomaly_points=60,  faulty_kpi_count=2, total_points_in_window=60,  earliest_anomaly_ts=2020-05-24 01:30:00+08:00\n"
    "- os_007: total_anomaly_points=31,  faulty_kpi_count=1, total_points_in_window=31,  earliest_anomaly_ts=2020-05-24 01:30:00+08:00\n"
    "\n"
    "Top KPI contributors for the top 3 nodes (kpi_name(anomaly_count), up to 5 each):\n"
    "- os_018: Agent_ping(30), Disk_rd_ios(30), Disk_io_util(30), Disk_await(29)\n"
    "- os_001: Disk_rd_kbs(30), ICMP_ping(30), Disk_rd_ios(30), Page_po(14), ss_total(7)\n"
    "- os_002: ICMP_ping(30), Disk_io_util(30), Agent_ping(30)\n"
    "\n"
    "Notes:\n"
    "- Many disk and ping-related KPIs are the primary contributors to anomaly points on these nodes.\n"
    "- Earliest anomalies for the top nodes occurred at the start of the incident window (2020-05-24 01:30:00+08:00).\n"
    "- If you want the full per-node table or to export the top-20 rows, inspect 'agg_by_cmdb' and 'top_kpis_for_top3'."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  total_anomaly_points  faulty_kpi_count  total_points_in_window       earliest_anomaly_ts
0   os_018                   119                 4                     120 2020-05-24 01:30:00+08:00
1   os_001                   111                 5                     150 2020-05-24 01:30:00+08:00
2   os_002                    90                 3                      90 2020-05-24 01:30:00+08:00
3   os_017                    60                 2                      60 2020-05-24 01:30:00+08:00
4   os_007                    31                 1                      31 2020-05-24 01:30:00+08:00
5   os_020                    29                 1                      31 2020-05-24 01:30:00+08:00
6  osb_001                    28                 1                      31 2020-05-24 01:30:00+08:00
7   os_015                    14                 1                      31 2020-05-24 01:30:00+08:00
8   db_007                    13                 1                      31 2020-05-24 01:31:00+08:00
9   os_012                    12                 1                      31 2020-05-24 01:33:00+08:00,    cmdb_id      kpi_name  anomaly_count
0   os_001   Disk_rd_kbs             30
1   os_001     ICMP_ping             30
2   os_001   Disk_rd_ios             30
3   os_001       Page_po             14
4   os_001      ss_total              7
5   os_002     ICMP_ping             30
6   os_002  Disk_io_util             30
7   os_002    Agent_ping             30
8   os_018    Agent_ping             30
9   os_018   Disk_rd_ios             30
10  os_018  Disk_io_util             30
11  os_018    Disk_await             29, 'os_018: Agent_ping(30), Disk_rd_ios(30), Disk_io_util(30), Disk_await(29) | os_001: Disk_rd_kbs(30), ICMP_ping(30), Disk_rd_ios(30), Page_po(14), ss_total(7) | os_002: ICMP_ping(30), Disk_io_util(30), Agent_ping(30)')```
```

In [ ]:
# In[5]:

# Prepare os_018 KPI compact table combining global stats and in-window stats
import pandas as pd
import numpy as np
import pytz

# Reuse variables from previous steps where possible
try:
    anomalies_summary
    metric_aggregates
    df_metric
    tz
    window_start
    window_end
except NameError:
    # Fallback definitions if not present (should not be needed in stateful kernel)
    tz = pytz.timezone('Asia/Shanghai')
    window_start = pd.Timestamp('2020-05-24 01:30:00', tz=tz)
    window_end = pd.Timestamp('2020-05-24 02:00:00', tz=tz)
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_aggregates = pd.DataFrame()  # placeholder
    anomalies_summary = pd.DataFrame()  # placeholder

# Get KPI names for os_018 that were flagged
os018_kpis = anomalies_summary[anomalies_summary['cmdb_id'] == 'os_018']['kpi_name'].unique().tolist()

# If none, produce empty results
if len(os018_kpis) == 0:
    os018_kpi_table = pd.DataFrame(columns=[
        'kpi_name','global_p95','global_p5','global_mean','global_std',
        'in_window_min','in_window_max','in_window_mean','anomaly_count','earliest_anomaly_ts','relative_breach'
    ])
    summary_line = "No flagged KPIs for cmdb_id=os_018 in anomalies_summary."
else:
    # Global stats for these KPIs from metric_aggregates
    global_stats = metric_aggregates[
        (metric_aggregates['cmdb_id'] == 'os_018') &
        (metric_aggregates['kpi_name'].isin(os018_kpis))
    ][['kpi_name','p95','p5','mean','std']].rename(
        columns={'p95':'global_p95','p5':'global_p5','mean':'global_mean','std':'global_std'}
    )

    # Filter metric data to os_018 and incident window
    df_os18_window = df_metric[
        (df_metric['cmdb_id'] == 'os_018') &
        (df_metric['ts'] >= window_start) &
        (df_metric['ts'] <= window_end) &
        (df_metric['kpi_name'].isin(os018_kpis))
    ].copy()

    # Merge global thresholds into windowed data
    df_os18_window = df_os18_window.merge(global_stats, on='kpi_name', how='left')

    # Use global thresholds to flag anomalies (as required by rule #9)
    df_os18_window['is_anomaly'] = (df_os18_window['value'] >= df_os18_window['global_p95']) | \
                                   (df_os18_window['value'] <= df_os18_window['global_p5'])
    df_os18_window['anomaly_ts'] = df_os18_window['ts'].where(df_os18_window['is_anomaly'], pd.NaT)

    # Aggregate in-window stats per KPI
    in_window_agg = df_os18_window.groupby('kpi_name').agg(
        in_window_min=('value','min'),
        in_window_max=('value','max'),
        in_window_mean=('value','mean'),
        anomaly_count=('is_anomaly','sum'),
        earliest_anomaly_ts=('anomaly_ts','min'),
        total_points_in_window=('value','count')
    ).reset_index()

    # Merge with global stats to form final table
    os018_kpi_table = in_window_agg.merge(global_stats, on='kpi_name', how='left')

    # Compute relative breach safely: (in_window_max - global_p95) / global_p95, NaN if global_p95 == 0 or missing
    os018_kpi_table['relative_breach'] = np.where(
        (os018_kpi_table['global_p95'].notna()) & (os018_kpi_table['global_p95'] != 0),
        (os018_kpi_table['in_window_max'] - os018_kpi_table['global_p95']) / os018_kpi_table['global_p95'],
        np.nan
    )

    # Keep only KPIs that were flagged in anomalies_summary (should already be the case) and anomaly_count>0
    os018_kpi_table = os018_kpi_table[os018_kpi_table['anomaly_count'] > 0].copy()

    # Round numeric columns to >=4 decimals
    for c in ['global_p95','global_p5','global_mean','global_std','in_window_min','in_window_max','in_window_mean','relative_breach']:
        if c in os018_kpi_table.columns:
            os018_kpi_table[c] = os018_kpi_table[c].round(4)

    # Select and order columns as requested, limit to 20 rows
    os018_kpi_table = os018_kpi_table[[
        'kpi_name','global_p95','global_p5','global_mean','global_std',
        'in_window_min','in_window_max','in_window_mean','anomaly_count','earliest_anomaly_ts','relative_breach'
    ]].sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)

    # Determine KPI with largest relative breach (exclude NaN)
    valid_rb = os018_kpi_table.dropna(subset=['relative_breach'])
    if not valid_rb.empty:
        idx = valid_rb['relative_breach'].idxmax()
        top_kpi = valid_rb.loc[idx, 'kpi_name']
        top_rb_val = valid_rb.loc[idx, 'relative_breach']
        summary_line = f"Largest relative breach on os_018: {top_kpi} with relative_breach={top_rb_val:.4f}"
    else:
        summary_line = "Relative breach undefined for os_018 KPIs (global_p95 == 0 for all), cannot determine largest relative breach."

# Display compact table and the one-line summary
os018_kpi_table, summary_line

```
Out[5]:
```
summary_lines = []
summary_lines.append("os_018 KPI anomaly summary (incident window 2020-05-24 01:30:00+08 -> 2020-05-24 02:00:00+08):")
for r in os018_kpi_table.itertuples(index=False):
    k = r.kpi_name
    gp95 = f"{r.global_p95:.4f}" if pd.notna(r.global_p95) else "nan"
    gp5 = f"{r.global_p5:.4f}" if pd.notna(r.global_p5) else "nan"
    gmean = f"{r.global_mean:.4f}" if pd.notna(r.global_mean) else "nan"
    gstd = f"{r.global_std:.4f}" if pd.notna(r.global_std) else "nan"
    iw_min = f"{r.in_window_min:.4f}" if pd.notna(r.in_window_min) else "nan"
    iw_max = f"{r.in_window_max:.4f}" if pd.notna(r.in_window_max) else "nan"
    iw_mean = f"{r.in_window_mean:.4f}" if pd.notna(r.in_window_mean) else "nan"
    ac = int(r.anomaly_count)
    ets = r.earliest_anomaly_ts
    rb = f"{r.relative_breach:.4f}" if pd.notna(r.relative_breach) else "nan"
    summary_lines.append(
        f"- {k}: anomalies={ac}, earliest={ets}, global_p95={gp95}, p5={gp5}, mean={gmean}, std={gstd}; "
        f"in_window_min={iw_min}, in_window_max={iw_max}, in_window_mean={iw_mean}; relative_breach={rb}"
    )

summary_lines.append("")
summary_lines.append(summary_line)

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       kpi_name  global_p95  global_p5  global_mean  global_std  in_window_min  in_window_max  in_window_mean  anomaly_count       earliest_anomaly_ts  relative_breach
0    Agent_ping      1.0000        1.0       1.0000      0.0000            1.0           1.00           1.000             30 2020-05-24 01:30:00+08:00            0.000
1  Disk_io_util      0.0000        0.0       0.0000      0.0000            0.0           0.00           0.000             30 2020-05-24 01:30:00+08:00              NaN
2   Disk_rd_ios      0.0000        0.0       0.0000      0.0000            0.0           0.00           0.000             30 2020-05-24 01:30:00+08:00              NaN
3    Disk_await     20.6425        0.0       3.0413      6.9147            0.0          24.11           1.488             29 2020-05-24 01:30:00+08:00            0.168, 'Largest relative breach on os_018: Disk_await with relative_breach=0.1680')```
```